# Un chiffre tombe, tout se réajuste · *One number lands, everything reprices*

Notebook compagnon du chapitre **1. Qu'est-ce que la macroéconomie financière ?** — [lire l'article](https://nmlab.io/ressources/qu-est-ce-que-la-macroeconomie-financiere).
Companion notebook to chapter **1. What Is Financial Macroeconomics? When the Economy Meets the Markets** — [read the article](https://nmlab.io/en/ressources/what-is-financial-macroeconomics).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import numpy as np
from numpy import ndarray

def simulate_reaction(pre: float, post: float, drift: float, pre_noise: float,
                      post_noise: float, scale: float, seed: int) -> tuple[ndarray, ndarray]:
    """Scène stylisée : une série bruitée qui saute au moment de la publication (t=0).
    Stylized scene: a noisy series that steps at the release (t=0).
    pre -> post est la marche ; drift ajoute une lente dérive post-publication."""
    rng = np.random.default_rng(seed)
    t = np.linspace(-30, 60, 271)
    step = 1.0 / (1.0 + np.exp(-t / scale))
    base = pre + (post - pre) * step + drift * np.clip(t, 0, None) / 60.0
    noise = np.where(t < 0, pre_noise, post_noise) * rng.standard_normal(t.size)
    noise = np.convolve(noise, np.ones(5) / 5, mode="same")
    return t, base + noise

futures = simulate_reaction(0.0, -0.78, -0.02, 0.020, 0.014, 3.0, seed=1)
yields = simulate_reaction(4.195, 4.253, -0.045, 0.006, 0.006, 1.4, seed=7)
dollar = simulate_reaction(0.0, 0.385, -0.03, 0.012, 0.010, 1.2, seed=3)


from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="Un chiffre tombe, tout se réajuste",
        sub="Premier vendredi du mois, 14 h 30 à Paris : les créations d'emplois américaines",
        panels=["Contrats à terme sur indice", "Rendement à 10 ans", "Dollar face à l'euro"],
        release="publication",
        xticklabels=["−30 s", "0", "+30 s", "+60 s"],
        note="Scène stylisée (données simulées) : le sens et l'ampleur des mouvements dépendent\n"
             "de l'écart entre le chiffre et le consensus — le monde concret, lui, n'a pas bougé."),
    "en": dict(
        title="One number lands, everything reprices",
        sub="First Friday of the month, 8:30 a.m. in New York: the U.S. job creations",
        panels=["Index futures", "10-year yield", "Dollar against the euro"],
        release="release",
        xticklabels=["−30 s", "0", "+30 s", "+60 s"],
        note="Stylized scene (simulated data): the direction and size of the moves depend on the gap\n"
             "between the number and the consensus — the tangible world has not moved."),
}

def pct(decimals: int, lang: str) -> FuncFormatter:
    """Formateur d'axe en pourcentage, virgule décimale en français."""
    def fmt(v, _):
        s = f"{v:.{decimals}f} %"
        return s.replace(".", ",") if lang == "fr" else s
    return FuncFormatter(fmt)

def build_figure(series: list[tuple], lang: str) -> Figure:
    """Trois panneaux : contrats à terme, rendement à 10 ans, dollar — autour de t=0."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1026)
    specs = [
        dict(data=series[0], color=nm.COLORS["rose"], ylim=(-0.95, 0.30),
             yticks=[-0.8, -0.4, 0.0, 0.2], dec=1),
        dict(data=series[1], color=nm.COLORS["blue"], ylim=(4.14, 4.315),
             yticks=[4.15, 4.20, 4.25, 4.30], dec=2),
        dict(data=series[2], color=nm.COLORS["blue"], ylim=(-0.28, 0.50),
             yticks=[-0.2, 0.0, 0.2, 0.4], dec=1),
    ]
    lefts, width = [0.075, 0.396, 0.717], 0.265
    for i, sp in enumerate(specs):
        ax = fig.add_axes([lefts[i], 0.195, width, 0.452])
        t, y = sp["data"]
        ax.axvline(0, color=nm.COLORS["muted"], linestyle=(0, (5, 4)), linewidth=1.8, alpha=0.9)
        ax.plot(t, y, color=sp["color"], linewidth=2.6)
        ax.set_xlim(-30, 60)
        ax.set_xticks([-30, 0, 30, 60])
        ax.set_xticklabels(text["xticklabels"])
        ax.set_ylim(*sp["ylim"])
        ax.set_yticks(sp["yticks"])
        ax.yaxis.set_major_formatter(pct(sp["dec"], lang))
        ax.set_title(text["panels"][i], fontsize=25, fontweight="bold",
                     color=nm.COLORS["text"], pad=14)
    fig.axes[1].text(-27, 4.307, text["release"], fontsize=18, color=nm.COLORS["muted"],
                     va="top", ha="left")
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure([futures, yields, dollar], LANG)